# Telco Churn Modeling

This notebook will build churn prediction models and export churn probabilities.

## Workflow
1. Load engineered features
2. Build and evaluate classification models
3. Generate per-customer churn risk scores for dashboarding

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features_df = pd.read_csv("../data/processed/telco_churn_features.csv")
full_df = pd.read_csv("../data/processed/telco_churn_processed.csv")

X = features_df.drop(columns=["ChurnFlag"])
y = features_df["ChurnFlag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

In [2]:
models = {
    "logistic_regression": LogisticRegression(max_iter=2000, solver="liblinear", class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    results.append({"model": name, "roc_auc": auc})
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

,model,roc_auc
0,logistic_regression,0.841652
1,random_forest,0.833718


In [3]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted[best_model_name]

risk_scores = best_model.predict_proba(X)[:, 1]

scores = pd.DataFrame({
    "customer_id": full_df["customerID"],
    "churn_actual": full_df["Churn"],
    "churn_risk_score": risk_scores,
})

scores.head(10)

,customer_id,churn_actual,churn_risk_score
0,7590-VHVEG,No,0.805855
1,5575-GNVDE,No,0.117704
2,3668-QPYBK,Yes,0.531932
3,7795-CFOCW,No,0.080768
4,9237-HQITU,Yes,0.854540
5,9305-CDSKC,Yes,0.908750
6,1452-KIOVK,No,0.719518
7,6713-OKOMC,No,0.524136
8,7892-POOKP,Yes,0.816489
9,6388-TABGU,No,0.033434


In [4]:
out_path = "../outputs/churn_scores_from_notebook.csv"
scores.sort_values("churn_risk_score", ascending=False).to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: ../outputs/churn_scores_from_notebook.csv
